# ProductIQ Phase 2 — Step 2.6: Deduplication

This notebook profiles duplication patterns and applies the conservative ProductIQ deduplication policy.

Only exact duplicate rows may be removed. Business-key, Product_URL, and URL_image duplicates are preserved.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from data.deduplication import (
    AjioDataDeduplicator,
    AjioDuplicationProfiler,
    DEDUPLICATION_POLICY_SUMMARY,
)
from data.ingestion import AjioDataLoader
from data.preprocessing import AjioDataCleaner, AjioDataNormalizer
from data.validation import AjioDataValidator


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    msg = "Could not locate ProductIQ project root"
    raise RuntimeError(msg)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATASET_PATH = PROJECT_ROOT / "resources" / "raw" / "Ajio Fashion Clothing.csv"

raw_df = AjioDataLoader().load(DATASET_PATH)
validation_report = AjioDataValidator().validate(raw_df)
cleaned_df = AjioDataCleaner().clean(raw_df).dataframe
normalized_df = AjioDataNormalizer().normalize(cleaned_df).dataframe

print("Rows raw:", len(raw_df))
print("Rows cleaned:", len(cleaned_df))
print("Rows normalized:", len(normalized_df))

In [ ]:
profiler = AjioDuplicationProfiler()
profile = profiler.profile(normalized_df)
print(profile.summary())

## Deduplication policy

ProductIQ currently treats `Id_Product` as the stable catalog-record identifier. Business-attribute similarity is not sufficient evidence to merge records because the source dataset does not expose an explicit parent-product or variant relationship.

Rules:

1. Remove exact duplicate rows only (keep first occurrence).
2. Treat conflicting duplicate `Id_Product` values as an identity violation.
3. Preserve Product_URL, URL_image, and business-attribute duplicate patterns.

In [ ]:
print(DEDUPLICATION_POLICY_SUMMARY)

deduplicator = AjioDataDeduplicator()
deduplication_result = deduplicator.deduplicate(normalized_df)
deduplicated_df = deduplication_result.dataframe
deduplication_report = deduplication_result.report

print("\nBefore:", len(normalized_df))
print("After:", len(deduplicated_df))
print(deduplication_report.summary())

In [ ]:
url_dupes = normalized_df["Product_URL"].value_counts()
url_dupes = url_dupes[url_dupes > 1]
print("Duplicate Product_URL values preserved:", len(url_dupes))
print("Rows on duplicated Product_URL values:", int(url_dupes.sum()))
print("Rows removed on AJIO dataset:", deduplication_report.rows_removed)

In [ ]:
business_key = normalized_df.groupby(
    ["Brand", "Description", "Category_by_gender", "Color"],
    dropna=False,
).size().reset_index(name="row_count")
business_dupes = business_key[business_key["row_count"] > 1]
print("Business-key duplicate groups preserved:", len(business_dupes))
print("Rows in those groups:", int(business_dupes["row_count"].sum()))
print(
    "These records remain because business attributes alone do not establish product identity."
)